In [10]:
import torch
import warp as wp
import numpy as np
import torch
from icecream import ic
from torch import optim


In [11]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [12]:
def rosenbrock(x: torch.tensor, y: torch.tensor):
    return (1.0 - x) ** 2.0 + 100.0 * (y - x**2.0) ** 2.0

In [13]:

a = torch.tensor(2.0, requires_grad=True, device=device)
ic(a)
b = torch.tensor(1.0, requires_grad=True, device=device)
ic(b)
print(a.grad)
result = rosenbrock(a,b)
ic(result)

ic| a: tensor(2., device='cuda:0', requires_grad=True)
ic| b: tensor(1., device='cuda:0', requires_grad=True)
ic| result: tensor(901., device='cuda:0', grad_fn=<AddBackward0>)


None


tensor(901., device='cuda:0', grad_fn=<AddBackward0>)

In [14]:
result.backward()

In [15]:
print(a.grad)
ic(a.dtype)
ic(a.device)
print(a.grad_fn)  

ic| a.dtype: torch.float32
ic| a.device: device(type='cuda', index=0)


tensor(2402., device='cuda:0')
None


In [16]:
grad_fns = [(result.grad_fn, 0)]
curr_level = 0
lines = []
while grad_fns:
    prev_level = curr_level
    fn, curr_level = grad_fns.pop()
    if curr_level != prev_level:
        print("---")
    print(fn.name())
    for next_fn, _ in fn.next_functions:
        if next_fn:
            grad_fns.append((next_fn, curr_level + 1))

AddBackward0
---
MulBackward0
---
PowBackward0
---
SubBackward0
---
PowBackward0
---
torch::autograd::AccumulateGrad
---
torch::autograd::AccumulateGrad
---
PowBackward0
---
RsubBackward1
---
torch::autograd::AccumulateGrad


In [19]:
def rosenbrock(x: torch.Tensor, y: torch.Tensor):
    return (1.0 - x)**2 + 100.0 * (y - x**2)**2

# 初始化参数（必须 requires_grad=True）
a = torch.tensor(2.0, requires_grad=True)
b = torch.tensor(1.0, requires_grad=True)

# 把参数打包进 optimizer
optimizer = torch.optim.Adam([a, b], lr=0.2)

# 优化循环
for i in range(10000):
    optimizer.zero_grad()           # 清空之前的梯度
    loss = rosenbrock(a, b)         # 前向计算
    loss.backward()                 # 反向传播
    optimizer.step()                # Adam 自动更新参数
    #print(a.grad_fn)  
    # if i % 100 == 0 or i == 999:
    #     ic(i, loss.item(), a.item(), b.item())
